# Hyperparameter sweep protocol

As of 2026-10-25.

Every model is tuned on the same grid: 5 readouts × 5 message-passing depths × 6 dropout rates, each config run on 5 seeds, with the config chosen on validation and reported on test.

## Search space

The sweep is a full grid, not random search: 5 depths × 6 dropout rates = 30 configs per (dataset, model, readout).

| Hyperparameter | Values | Count | Notes |
| --- | --- | --- | --- |
| Readout | max, mean, sum, gated attention, multi-head attention | up to 5 | Only the readouts a model supports (next section) |
| Message-passing layers | 2, 3, 4, 5, 6 | 5 | Count of message-passing steps only; readout and prediction head excluded |
| Dropout | 0.0, 0.1, 0.2, 0.3, 0.4, 0.5 | 6 | One rate for the whole model, wherever the model applies dropout |

"Layers" maps to each model's own argument: `num_layers` for GIN/GINE, `depth` for Chemprop, `num_layers` for AttentiveFP (which counts its first gated-attention layer).

In [1]:
# Sweep grid: the single source for the numbers in this protocol.
LAYERS   = [2, 3, 4, 5, 6]
DROPOUT  = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5]
SEEDS    = [0, 1, 2, 3, 4]
DATASETS = ['BBBP', 'BACE', 'CLINTOX', 'ESOL', 'FREESOLV', 'LIPO']
READOUTS = {   # --readout names each model supports (AttentiveFP: to confirm)
    'gin':      ['max', 'mean', 'sum', 'attention', 'multihead'],
    'gine':     ['max', 'mean', 'sum', 'attention', 'multihead'],
    'chemprop': ['max', 'mean', 'sum', 'attention'],
}
FIXED = dict(epochs=300, patience=50, hidden_dim=128, lr=1e-3, weight_decay=1e-5)

configs = [(layers, dropout) for layers in LAYERS for dropout in DROPOUT]
print(f"{len(configs)} configs x {len(SEEDS)} seeds = {len(configs) * len(SEEDS)} runs per (dataset, model, readout)")

30 configs x 5 seeds = 150 runs per (dataset, model, readout)


## Readouts per model

Each model is swept over the readouts it already supports; a readout a model lacks is skipped, not built for the sweep. Cells give the name passed as `--readout`.

| Model | Max | Mean | Sum | Gated attention | Multi-head attention |
| --- | --- | --- | --- | --- | --- |
| GIN | `max` | `mean` | `sum` | `attention` | `multihead` (to be added) |
| GINE | `max` | `mean` | `sum` | `attention` | `multihead` (to be added) |
| Chemprop | `max` (repo's own, see below) | `mean` | `sum` | `attention` (Chemprop `AttentiveAggregation`) | Not available |
| AttentiveFP | To confirm | To confirm | To confirm | Built-in attentive readout | Not available |

- **Multi-head attention for GIN/GINE** = PyG's `SetTransformerAggregation` (attention pooling with one learned seed vector, 4 heads). It is not in the code yet; it gets wired in as readout `multihead` before the sweep starts.
- **Chemprop readouts, where each comes from** (checked against Chemprop 2.2.1 and `src/chemprop_model.py`):
    - `mean`, `sum`: Chemprop built-ins. Chemprop's own CLI (`chemprop train --aggregation`) offers only these plus `norm` (its default, a sum divided by a constant).
    - `attention`: Chemprop's `AttentiveAggregation` class. It ships with Chemprop but is not selectable from its CLI, only through the Python API, which is how this repo builds the model.
    - `max`: not part of Chemprop. `src/chemprop_model.py` defines its own `MaxAggregation` (an element-wise max over atom embeddings).
    - None of the four has been run in this environment yet (Chemprop is not installed in `graph-pooling`); the Chemprop smoke test must cover all four before the sweep.
- **Do not use `selfattention`** (the repo's single-head readout). It reads only the first atom's output, which leaks the label on BBBP (0.99 test AUC).
- **AttentiveFP:** whoever adds it fills its row: whether plain max/mean/sum can replace its built-in readout depends on how the encoder and readout are split.

## Fixed settings

Everything outside the search space is held constant so that differences come from the swept hyperparameters alone.

| Setting | Value |
| --- | --- |
| Max epochs | 300 |
| Early stopping | Patience 50 epochs on the validation metric; the best-validation checkpoint is reloaded for final scoring |
| Validation metric | Binary: ROC-AUC (ClinTox: mean of its 2 per-task AUCs). Regression: RMSE |
| Loss | Binary: BCE. Regression: MSE on targets standardized with train-split mean/std; metrics reported in original units |
| Hidden size | 128 |
| Optimizer | Adam, learning rate 1e-3, weight decay 1e-5 (Chemprop keeps its built-in warmup/decay schedule peaking at 1e-3) |
| Batch size | Per dataset config: BBBP 64, BACE 32, ClinTox 32, ESOL 32, FreeSolv 32, Lipophilicity 64 |
| Features | PyG MoleculeNet defaults (9 atom, 3 bond features); Chemprop uses its own featurizer |
| Split | Scaffold split, 80/10/10, class-stratified for binary datasets |

Patience 50 because 100-epoch base runs were often still improving at the cap (Lipophilicity: mean best epoch 72, several at 100). The dataset configs still say 100 epochs and patience 20; the sweep overrides both.

## Seeds and splits

Every config runs on seeds 0, 1, 2, 3 and 4, and every model uses the same 5 split files.

- The seed sets both the scaffold split (which scaffold groups go to val vs test) and the model's initialization and batch order. Spread across seeds therefore includes split-to-split variation.
- Splits live in `data/splits/<DATASET>_scaffold_seed<N>.json`. Use the shared copies; do not regenerate them on your own machine. Seeds 0–2 already exist from the base runs; 3–4 are written on first use, once, before the sweep fans out.
- Chemprop reads the same split files, so all models see identical molecules in each split.
- Because splits are shared, any two configs, readouts or models can be compared seed by seed (paired).

## Selection and reporting

Configs are chosen on validation averaged over all 5 seeds; test is only read for the chosen config.

1. For each (dataset, model, readout), average the validation metric of each of the 30 configs over the 5 seeds.
2. Pick the config with the best average: highest AUC or lowest RMSE. If two are within 0.001, take fewer layers, then lower dropout.
3. Report that config's test metric as mean ± std and 95% CI over the 5 seeds (t-distribution, 4 degrees of freedom), with its layers, dropout and mean best epoch.
4. Never use test results to choose a config, a readout or an epoch.
5. A seed whose validation metric is NaN (val split with one class) counts as missing. A config needs at least 4 of 5 valid seeds to be eligible; report n.
6. To name a model's best readout, compare the readouts' selected configs on mean validation, then report that readout's test.
7. Compare readouts within a model with a paired t-test on test metric across the 5 seeds, Benjamini-Hochberg corrected within each (dataset, model). Report Wilcoxon too, but note that with 5 pairs it cannot go below p = 0.0625.

The headline table is one row per (model, readout) and one column per dataset, each cell "test mean ± std (layers, dropout)".

## Run budget

The full grid is 150 runs per (dataset, model, readout), about 12,600 runs for GIN, GINE and Chemprop together.

| Model | Readouts | Runs (6 datasets × readouts × 30 configs × 5 seeds) |
| --- | --- | --- |
| GIN | 5 | 4,500 |
| GINE | 5 | 4,500 |
| Chemprop | 4 | 3,600 |
| AttentiveFP | 1–4 (to confirm) | 900–3,600 |

- **Time per run (rough):** in the 100-epoch base runs on a single gpu-preempt GPU, GIN/GINE took about 15 s (BBBP) to 40 s (Lipophilicity) per run. With up to 300 epochs and up to 6 layers, expect roughly 30 s to 3 min, so about 50–150 GPU-hours per PyG model. Chemprop has not been timed.
- **One job per (dataset, model, readout):** 150 runs in sequence on 1 GPU, 4 CPUs, 16 GB, 8 h walltime. That is 30 jobs per model (24 for Chemprop); submit as an array capped at 10–20 running at once.
- **Never one job per run.** Thousands of sub-minute jobs overload the scheduler and spend more time queueing than training.
- **Resumable:** on gpu-preempt, jobs get requeued. A job must skip runs whose metrics file already exists rather than starting over.

In [2]:
import pandas as pd

budget = pd.DataFrame([
    {'model': m, 'readouts': len(r),
     'runs': len(DATASETS) * len(r) * len(configs) * len(SEEDS),
     'jobs (one per dataset x readout)': len(DATASETS) * len(r)}
    for m, r in READOUTS.items()
])
budget.loc[len(budget)] = ['total', '', budget['runs'].sum(), budget['jobs (one per dataset x readout)'].sum()]
budget

,model,readouts,runs,jobs (one per dataset x readout)
0,gin,5,4500,30
1,gine,5,4500,30
2,chemprop,4,3600,24
3,total,,12600,84


## Output format

Each config gets its own folder, because `run_single.py` names files by dataset, model, readout and seed only and would otherwise overwrite across configs.

```
results/sweep/<DATASET>/<model>/<readout>/L<layers>_do<dropout>/<DATASET>_<model>_<readout>_seed<N>_metrics.json
```

For example `results/sweep/ESOL/gin/sum/L4_do0.2/ESOL_gin_sum_seed3_metrics.json`.

Every metrics file must contain:

| Field | Example |
| --- | --- |
| `dataset`, `model`, `readout`, `seed`, `task_type` | `ESOL`, `gin`, `sum`, `3`, `regression` |
| `num_layers`, `dropout`, `hidden_dim`, `lr`, `batch_size` | `4`, `0.2`, `128`, `0.001`, `32` |
| `epochs`, `patience` | `300`, `50` |
| `best_epoch` | `117` |
| Train/val/test metrics | Binary: `train_auc`, `val_auc`, `test_auc`. Regression: `train_rmse`, `val_rmse`, `test_rmse` (+ `_mae`, `_r2`) |

The current metrics files already carry everything except `epochs` and `patience`. The sweep produces two tables: `sweep_results.csv` (one row per run) and `sweep_selected.csv` (one row per dataset, model and readout with the chosen config and its test mean, std, CI and n).

In [3]:
def run_dir(dataset, model, readout, layers, dropout):
    """Folder for one config; holds one metrics file per seed."""
    return f"results/sweep/{dataset}/{model}/{readout}/L{layers}_do{dropout}"

run_dir('ESOL', 'gin', 'sum', 4, 0.2)

'results/sweep/ESOL/gin/sum/L4_do0.2'

## Checks

Run a smoke test before submitting anything large, and audit the results before reporting them.

**Before launch**

- [ ] Smoke test each model: 1 dataset per task type, 1 config, 1 seed, 2 epochs; check the metrics file has every required field.
- [ ] Split files for seeds 0–4 exist for all 6 datasets in the shared `data/splits/`.
- [ ] Grid count matches the run budget table for your model before submitting.

**After the runs finish**

- [ ] Every (dataset, model, readout) has 150 metrics files; rerun gaps rather than ignoring them.
- [ ] Share of runs whose best epoch is above 250 (stopped by the 300 cap, not early stopping). If it is large for a dataset, flag it rather than silently raising epochs.
- [ ] Count of NaN validation or test metrics, per dataset.
- [ ] Chosen configs at the edge of the grid (2 or 6 layers, dropout 0.0 or 0.5): flag them; the optimum may lie outside the range.

## Known pitfalls

These came up in the base runs and affect how sweep results should be read.

| Issue | Effect | What to do |
| --- | --- | --- |
| BBBP first-atom shortcut | The first atom of each SMILES alone predicts the label at 0.979 test AUC (e.g. all 476 molecules starting `C1=C` are positive) | Any readout that reads a single node is invalid on BBBP; check new readouts are permutation invariant |
| FreeSolv test split is ~52 molecules | Test RMSE varies by more than ±1 across seeds | Report CIs; do not rank models on FreeSolv alone |
| ClinTox has 2 tasks | One task (toxicity) is ~8% positive | The reported AUC is the mean of the two per-task AUCs; splits keep both classes in every split |
| Scaffold split shifts targets | ESOL val/test molecules average ~1 log unit less soluble than train | Expect a train-test gap on regression; it is not a bug |
| GPU nondeterminism | Identical runs differ (ESOL RMSE moved ~11% across repeats) | Treat differences within seed spread as noise; never compare single runs |
| Raw integer atom features | Models without normalization layers trained poorly in base runs (GCN, GAT near chance on BACE) | GIN/GINE are fine (BatchNorm); check any new model trains above chance before sweeping it |

## Extending the sweep

Additions keep the same seeds, splits, selection rule and output format, so old and new results stay comparable.

- **New hyperparameter:** add a row to the search-space table with its values, remove it from fixed settings, add it to the folder name (e.g. `L4_do0.2_lr1e-3`) and to the required metrics fields. Note that the grid size multiplies; consider fixing other values first if the run budget explodes.
- **New model:** add a row to the readouts table and the run budget table, say which argument "layers" maps to, and pass the smoke test before sweeping.
- **New readout:** add a column to the readouts table with its `--readout` name per model, and confirm it is permutation invariant (see the BBBP shortcut).
- **New dataset:** add its batch size to fixed settings, create seeds 0–4 split files once, and add it to the run budget.

Record every change to this protocol with its date, so results can be matched to the protocol version they were run under.